# WtDtPorter 架构

```mermaid
graph TD
    subgraph "外部调用层 (Python/C#)"
        A["<b>外部应用</b><br/>调用C接口, 实现回调函数"]
    end

    subgraph "WtDtPorter 动态库"
        direction LR
        
        subgraph "<b>API接口层</b>"
            B(<b>WtDtPorter.h/.cpp</b>):::api_layer
            B_Desc["<b>作用:</b> 暴露C语言API<br/>作为外部调用的唯一入口<br/>将所有请求转发给WtDtRunner"]
        end

        subgraph "<b>核心协调层</b>"
            C(<b>WtDtRunner.h/.cpp</b>):::runner_layer
            C_Desc["<b>作用:</b> 系统总指挥 (单例)<br/>1. 初始化所有核心模块 (DataManager等)<br/>2. 管理扩展模块的生命周期<br/>3. 持有并触发外部回调函数"]
        end

        subgraph "<b>扩展桥接层</b>"
            D(<b>PorterDefs.h</b>):::defs_layer
            D_Desc["<b>作用:</b> 定义回调函数类型<br/>是C++核心与外部回调之间的'契约'"]
            E(<b>ExpParser.h/.cpp</b>):::bridge_layer
            E_Desc["<b>作用:</b> '代理'行情解析器<br/>将内部订阅请求<br/>转换为外部回调"]
            F(<b>ExpDumper.h/.cpp</b>):::bridge_layer
            F_Desc["<b>作用:</b> '代理'数据存储器<br/>将内部数据转储请求<br/>转换为外部回调"]
        end
    end

    %% --- 关系连线 ---
    A -- "调用API" --> B
    B -- "转发请求" --> C
    
    C -- "创建并管理" --> E & F

    E -- "将内部调用<br/>(如subscribe)" --> C
    F -- "将内部调用<br/>(如dumpHisBars)" --> C
    
    C -- "触发外部回调" --> A

    B -- "包含定义" --> D
    E -- "包含定义" --> D
    F -- "包含定义" --> D
    C -- "使用" --> E & F

    %% 样式定义
    classDef api_layer fill:#c9daf8,stroke:#333,stroke-width:2px
    classDef runner_layer fill:#d9ead3,stroke:#333,stroke-width:3px
    classDef bridge_layer fill:#fce5cd,stroke:#333,stroke-width:2px
    classDef defs_layer fill:#fff2cc,stroke:#333,stroke-width:2px
    classDef build_system fill:#eeeeee,stroke:#666,stroke-width:1px,stroke-dasharray: 5 5
```

# 调用外部代码接口协议 PorterDefs.h

定义了 WtDtPorter 与外部脚本语言（如 Python）之间的回调函数名称：供 WtDtPorter（C++）反向调用外部代码。
- **FuncParser\*** 回调：自定义行情源接入WonderTrader
  - `FuncParserEvtCallback`：typedef void\(PORTER_FLAG \*FuncParserEvtCallback\)\(WtUInt32 evtId, const char* id\);
    - 用于接收**生命周期事件**。C++核心会通过这个回调通知外部程序
      - evtId：事件类型，包括
          - `EVENT_PARSER_INIT` (1): Parser初始化事件：当Parser完成初始化时触发
          - `EVENT_PARSER_CONNECT` (2): Parser连接事件：当Parser成功连接到数据源时触发
          - `EVENT_PARSER_DISCONNECT` (3): Parser断开连接事件：当Parser与数据源断开连接时触发
          - `EVENT_PARSER_RELEASE` (4): Parser释放事件：当Parser释放资源时触发
      - id：Parser的唯一标识符，用于区分不同的Parser实例
  - `FuncParserSubCallback`：typedef void\(PORTER_FLAG \*FuncParserSubCallback\)\(const char* id, const char* fullCode, bool isForSub\);
    - 用于接收**行情订阅/退订请求**，当C++内部需要某个合约的行情时，`ParserAdapter`会触发这个回调。
      - id：Parser的唯一标识符，用于区分不同的Parser实例
      - fullCode：完整的合约代码，包含交易所前缀
      - isForSub：订阅标志，true表示订阅操作，false表示退订操作
- **FuncDump\*** 回调：数据自定义外部转储
  - `FuncDumpBars`、`FuncDumpTicks`、`FuncDumpOrdQue`、`FuncDumpOrdDtl`、`FuncDumpTrans`
  - 分别用于接收 **K线**、**Tick**、**委托队列**、**逐笔委托**、**逐笔成交**数据的批量转储任务
  - 当 StateMonitor 判断到盘后处理时间时，DataManager 会触发数据转储流程，WtDtCore 就会将数据通过这些回调函数到外部程序

# 扩展数据转储器 ExpDumper.h/cpp
```cpp
class ExpDumper : public IHisDataDumper
```
实现历史数据转储接口IHisDataDumper，支持K线、Tick、委托队列、委托明细、逐笔成交等多种数据类型的转储
- 参考 [Includes/note.ipynb/数据管理接口层/数据写入 IDataWriter.h/历史数据存储接口类 IHisDataDumper](../Includes/note.ipynb)

**成员**：
- `std::string	_id`：转储器唯一标识符，用于在回调函数中识别转储器实例

**全局声明**：
获取全局 `WtDtRunner` 单例的引用，该单例在 [./WtDtPorter.cpp](./WtDtPorter.cpp) 中定义
```cpp
extern WtDtRunner& getRunner();
```

## 方法

### 转储历史K线数据 dumpHisBars
```cpp
/**
 * @brief 转储历史K线数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param period K线周期，如"m1"、"m5"、"day"等
 * @param bars K线数据数组指针
 * @param count K线数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将K线数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的K线转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisBars(const char* stdCode, const char* period, WTSBarStruct* bars, uint32_t count)
{
	// 调用WtDtRunner的dumpHisBars方法，将转储器ID作为第一个参数传递
	// 便于回调函数识别数据来源和执行相应的存储逻辑
	return getRunner().dumpHisBars(_id.c_str(), stdCode, period, bars, count);
}
```

### 转储历史Tick数据 dumpHisTicks
```cpp
/**
 * @brief 转储历史Tick数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param ticks Tick数据数组指针
 * @param count Tick数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将Tick数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的Tick转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisTicks(const char* stdCode, uint32_t uDate, WTSTickStruct* ticks, uint32_t count)
{
	// 调用WtDtRunner的dumpHisTicks方法，将转储器ID作为第一个参数传递
	// 支持外部回调函数根据转储器ID执行不同的存储策略
	return getRunner().dumpHisTicks(_id.c_str(), stdCode, uDate, ticks, count);
}
```

### 转储历史委托队列数据 dumpHisOrdQue
```cpp
/**
 * @brief 转储历史委托队列数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param items 委托队列数据数组指针，包含买卖盘口队列信息
 * @param count 委托队列数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数重写了IHisDataDumper接口的dumpHisOrdQue方法，
 * 将委托队列数据转储请求转发给WtDtRunner进行处理。
 * 委托队列数据主要用于Level2行情分析。
 */
virtual bool dumpHisOrdQue(const char* stdCode, uint32_t uDate, WTSOrdQueStruct* items, uint32_t count) override;
```

### 转储历史委托明细数据 dumpHisOrdDtl
```cpp
/**
 * @brief 转储历史委托明细数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param items 委托明细数据数组指针
 * @param count 委托明细数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将委托明细数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的委托明细转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisOrdDtl(const char* stdCode, uint32_t uDate, WTSOrdDtlStruct* items, uint32_t count)
{
	// 调用WtDtRunner的dumpHisOrdDtl方法，将转储器ID作为第一个参数传递
	// 便于WtDtRunner进行转储器实例的识别和管理
	return getRunner().dumpHisOrdDtl(_id.c_str(), stdCode, uDate, items, count);
}
```

### 转储历史逐笔成交数据 dumpHisTrans
```cpp
/**
 * @brief 转储历史逐笔成交数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param items 逐笔成交数据数组指针
 * @param count 逐笔成交数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将逐笔成交数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的逐笔成交转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisTrans(const char* stdCode, uint32_t uDate, WTSTransStruct* items, uint32_t count)
{
	// 调用WtDtRunner的dumpHisTrans方法，将转储器ID作为第一个参数传递
	// 支持多个转储器同时工作时的识别和管理
	return getRunner().dumpHisTrans(_id.c_str(), stdCode, uDate, items, count);
}
```

# 扩展行情解析器 ExpParser.h/cpp
```cpp
class ExpParser : public IParserApi
```
实现行情解析器接口 IParserApi，支持初始化、连接、订阅、退订等基本操作
- 参考 [Includes/note.ipynb/行情解析 IParserApi.h/行情解析器接口 IParserApi](../Includes/note.ipynb)

**成员**：
- `std::string _id`：解析器唯一标识符，用于在回调函数中识别解析器实例
- `IParserSpi* m_sink`：回调接口指针，用于接收行情数据和事件
- `IBaseDataMgr* m_pBaseDataMgr`：基础数据管理器指针，用于访问合约、交易所等基础信息

**全局声明**：
获取全局 `WtDtRunner` 单例的引用，该单例在 [./WtDtPorter.cpp](./WtDtPorter.cpp) 中定义
```cpp
extern WtDtRunner& getRunner();
```

## 方法

### 初始化解析器 init
```cpp
/**
 * @brief 初始化解析器
 * @param config 配置参数，包含解析器的初始化配置信息
 * @return bool 初始化成功返回true
 * 
 * 该函数将初始化请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的初始化回调函数，通知外部模块进行初始化操作。
 * 当前实现总是返回true，表示初始化成功。
 */
bool ExpParser::init(WTSVariant* config)
{
	// 调用WtDtRunner的parser_init方法，将解析器ID传递给回调函数
	// 外部模块根据解析器ID进行相应的初始化操作
	getRunner().parser_init(_id.c_str());
	return true;  // 总是返回true，表示初始化成功
}
```

### 释放解析器资源 release
```cpp
/**
 * @brief 释放解析器资源
 * 
 * 该函数将释放请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的释放回调函数，通知外部模块清理资源。
 * 外部模块应在此回调中关闭连接、释放内存等清理操作。
 */
void ExpParser::release()
{
	// 调用WtDtRunner的parser_release方法，通知外部模块释放资源
	// 外部模块根据解析器ID识别需要释放的解析器实例
	getRunner().parser_release(_id.c_str());
}
```

### 连接到数据源 connect
```cpp
/**
 * @brief 连接到数据源
 * @return bool 连接成功返回true
 * 
 * 该函数将连接请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的连接回调函数，通知外部模块建立与数据源的连接。
 * 当前实现总是返回true，表示连接请求已发送。
 */
bool ExpParser::connect()
{
	// 调用WtDtRunner的parser_connect方法，通知外部模块建立连接
	// 外部模块根据解析器ID识别需要连接的数据源
	getRunner().parser_connect(_id.c_str());
	return true;  // 总是返回true，表示连接请求已发送
}
```

### 断开与数据源的连接 disconnect
```cpp
/**
 * @brief 断开与数据源的连接
 * @return bool 断开成功返回true
 * 
 * 该函数将断开连接请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的断开连接回调函数，通知外部模块关闭与数据源的连接。
 * 当前实现总是返回true，表示断开连接请求已发送。
 */
bool ExpParser::disconnect()
{
	// 调用WtDtRunner的parser_disconnect方法，通知外部模块断开连接
	// 外部模块根据解析器ID识别需要断开连接的数据源
	getRunner().parser_disconnect(_id.c_str());
	return true;  // 总是返回true，表示断开连接请求已发送
}
```

### 查询连接状态 isConnected
```cpp
/**
 * @brief 查询连接状态
 * @return bool 已连接返回true，未连接返回false
 * 
 * 该函数重写了IParserApi接口的isConnected方法。
 * 当前实现始终返回true，表示解析器处于连接状态。
 * 实际的连接状态由外部数据源维护。
 */
virtual bool isConnected() override { return true; }  // 始终返回true
```

### 订阅合约行情 subscribe
```cpp
/**
 * @brief 订阅合约行情
 * @param setCodes 要订阅的合约代码集合
 * 
 * 该函数将订阅请求转发给WtDtRunner处理。
 * 遍历合约集合，对每个合约代码调用WtDtRunner的订阅方法。
 * WtDtRunner会触发外部注册的订阅回调函数，通知外部模块向数据源发送订阅请求。
 */
void ExpParser::subscribe(const CodeSet& setCodes)
{
	// 遍历合约代码集合，对每个合约进行订阅
	for(const auto& code : setCodes)
		// 调用WtDtRunner的parser_subscribe方法，传递解析器ID和合约代码
		// 外部模块根据这些信息向数据源发送订阅请求
		getRunner().parser_subscribe(_id.c_str(), code.c_str());
}
```

### 退订合约行情 unsubscribe
```cpp
/**
 * @brief 退订合约行情
 * @param setCodes 要退订的合约代码集合
 * 
 * 该函数将退订请求转发给WtDtRunner处理。
 * 遍历合约集合，对每个合约代码调用WtDtRunner的退订方法。
 * WtDtRunner会触发外部注册的退订回调函数，通知外部模块向数据源发送退订请求。
 */
void ExpParser::unsubscribe(const CodeSet& setCodes)
{
	// 遍历合约代码集合，对每个合约进行退订
	for (const auto& code : setCodes)
		// 调用WtDtRunner的parser_unsubscribe方法，传递解析器ID和合约代码
		// 外部模块根据这些信息向数据源发送退订请求
		getRunner().parser_unsubscribe(_id.c_str(), code.c_str());
}
```

### 注册回调接口 registerSpi
```cpp
/**
 * @brief 注册回调接口
 * @param listener 回调接口指针，用于接收行情数据和事件
 * 
 * 该函数保存回调接口指针，并从回调接口获取基础数据管理器。
 * 回调接口用于向系统推送行情数据和事件。
 * 基础数据管理器用于访问合约、交易所等基础信息。
 */
void ExpParser::registerSpi(IParserSpi* listener)
{
	// 保存回调接口指针
	m_sink = listener;

	// 如果回调接口指针有效，则从回调接口获取基础数据管理器
	// 基础数据管理器用于访问合约、交易所、交易时段等基础信息
	if (m_sink)
		m_pBaseDataMgr = m_sink->getBaseDataMgr();
}
```

# WtDtRunner.h/cpp

## 成员
- `WTSBaseDataMgr	_bd_mgr`：基础数据管理器：管理合约、交易所、交易时段等基础信息
- `WTSHotMgr		_hot_mgr`：主力合约管理器：管理主力合约规则和次主力合约规则
- `boost::asio::io_service _async_io`：Boost异步IO服务：用于异步IO操作
- `StateMonitor	_state_mon`：状态监控器：监控交易时段状态，管理数据存储的打开和关闭
- `UDPCaster		_udp_caster`：UDP广播器：通过UDP协议广播行情数据
- `ShmCaster		_shm_caster`：共享内存广播器：通过共享内存广播行情数据
- `DataManager		_data_mgr`：数据管理器：管理行情数据的接收、处理、存储和分发
- `IndexFactory	_idx_factory`：指数工厂：管理指数的计算和发布
- `ParserAdapterMgr	_parsers`：行情解析器管理器：管理所有行情解析器的运行
- `bool _to_exit`：退出标志：true表示需要退出，false表示继续运行
---
- `FuncParserEvtCallback	_cb_parser_evt`：扩展Parser事件回调函数指针
- `FuncParserSubCallback	_cb_parser_sub`：扩展Parser订阅回调函数指针
---
- `FuncDumpBars	_dumper_for_bars`：K线数据转储回调函数指针
- `FuncDumpTicks	_dumper_for_ticks`：Tick数据转储回调函数指针
- `FuncDumpOrdQue	_dumper_for_ordque`：委托队列数据转储回调函数指针
- `FuncDumpOrdDtl	_dumper_for_orddtl`：委托明细数据转储回调函数指针
- `FuncDumpTrans	_dumper_for_trans`：逐笔成交数据转储回调函数指针
---
- `ExpDumpers		_dumpers`：扩展转储器映射表：管理所有扩展转储器实例
  - typedef std::map\<std::string, ExpDumperPtr\> ExpDumpers：扩展转储器映射表类型
  - typedef std::shared_ptr\<`ExpDumper`\> ExpDumperPtr：扩展转储器智能指针类型


## 方法

### 初始化与生命周期

#### 初始化数据服务 initialize
流程：
- 使用参数 logCfg 和 bLogCfgFile 对日志系统 WTSLogger 进行初始化
- 使用参数 modDir 通过 WtHelper 设置模块目录
- 通过 cfgFile 和 bCfgFile 加载基础数据文件配置到 configs
  - 读取 `basefiles`，使用其中的配置文件路径初始化 `_bd_mgr: WTSBaseDataMgr`、`_hot_mgr: WTSHotMgr`
  - 读取 `shmcaster`、`broadcaster` 来初始化 `_shm_caster: ShmCaster`、`_udp_caster: UDPCaster`，并设置 `_data_mgr: DataManager`
  - 读取 `allday` 来初始化 `_state_mon: StateMonitor`
  - 读取 `writer` 来初始化 `_data_mgr: DataManager`
  - 读取 `index`、`parsers` 来初始化 `_idx_factory: IndexFactory`、`_parsers: ParserAdapterMgr`
```cpp
/**
 * @brief 初始化数据服务
 * @param cfgFile 配置文件路径或配置内容字符串
 * @param logCfg 日志配置文件路径或配置内容字符串
 * @param modDir 模块目录路径，默认为空字符串（使用当前目录）
 * @param bCfgFile cfgFile是否为文件路径，默认为true
 * @param bLogCfgFile logCfg是否为文件路径，默认为true
 */
void WtDtRunner::initialize(const char* cfgFile, const char* logCfg, const char* modDir /* = "" */, bool bCfgFile /* = true */, bool bLogCfgFile /* = true */)
```

#### 启动数据服务 start

### 扩展行情解析器接口

#### 创建扩展行情解析器 createExtParser

#### 注册扩展Parser的回调函数 registerParserPorter

#### Parser初始化事件处理 parser_init

#### Parser连接事件处理 parser_connect

#### Parser释放事件处理 parser_release

#### Parser断开连接事件处理 parser_disconnect

#### Parser订阅处理 parser_subscribe

#### Parser退订处理 parser_unsubscribe

#### 处理扩展Parser推送的行情数据 on_ext_parser_quote

### 扩展数据转储器接口

#### 创建扩展数据转储器 createExtDumper

#### 注册扩展Dumper的回调函数 (K线和Tick) registerExtDumper

#### 注册扩展Dumper的回调函数 (高频数据) registerExtHftDataDumper

#### 转储历史K线数据 dumpHisBars

#### 转储历史Tick数据 dumpHisTicks

#### 转储历史委托队列数据 dumpHisOrdQue

#### 转储历史委托明细数据 dumpHisOrdDtl

#### 转储历史逐笔成交数据 dumpHisTrans

# WtDtPorter模块对外C语言接口 WtDtPorter.h/cpp

## 方法
好的，这是 `WtDtPorter.h` 和 `WtDtPorter.cpp` 中暴露的C语言API函数的分层代码段，按照您要求的 Markdown 格式：

- **基础接口 (Basic Interface)**
  - 初始化数据服务：`EXPORT_FLAG void initialize(WtString cfgFile, WtString logCfg, bool bCfgFile, bool bLogCfgFile)`
  - 启动数据服务：`EXPORT_FLAG void start(bool bAsync = false)`

- **辅助接口 (Utility Interface)**
  - 获取版本信息：`EXPORT_FLAG WtString get_version()`
  - 输出日志：`EXPORT_FLAG void write_log(unsigned int level, const char* message, const char* catName)`

- **扩展行情解析器接口 (Extended Parser Interface)**
  - 创建扩展行情解析器：`EXPORT_FLAG bool create_ext_parser(const char* id)`
  - 注册扩展Parser的回调函数：`EXPORT_FLAG void register_parser_callbacks(FuncParserEvtCallback cbEvt, FuncParserSubCallback cbSub)`
  - 向底层推送tick数据：`EXPORT_FLAG void parser_push_quote(const char* id, WTSTickStruct* curTick, WtUInt32 uProcFlag)`

- **扩展数据转储器接口 (Extended Dumper Interface)**
  - 创建扩展数据转储器：`EXPORT_FLAG bool create_ext_dumper(const char* id)`
  - 注册扩展Dumper的回调函数 (K线和Tick)：`EXPORT_FLAG void register_extended_dumper(FuncDumpBars barDumper, FuncDumpTicks tickDumper)`
  - 注册扩展Dumper的回调函数 (高频数据)：`EXPORT_FLAG void register_extended_hftdata_dumper(FuncDumpOrdQue ordQueDumper, FuncDumpOrdDtl ordDtlDumper, FuncDumpTrans transDumper)`

### 基础接口

#### 初始化数据服务 initialize

#### 启动数据服务 start

### 辅助接口

#### 获取版本信息 get_version

#### 输出日志 write_log

### 扩展行情解析器接口

#### 创建扩展行情解析器 create_ext_parser

#### 注册扩展Parser的回调函数 register_parser_callbacks

#### 向底层推送tick数据 parser_push_quote

### 扩展数据转储器接口

#### 创建扩展数据转储器 create_ext_dumper

#### 注册扩展Dumper的回调函数 (K线和Tick) register_extended_dumper

#### 注册扩展Dumper的回调函数 (高频数据) register_extended_hftdata_dumper

# ExpParser.h/cpp

# ExpDumper.h/cpp